# 🕺 Dancing Stick Figures v0.2 — train an image model, then a video model

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sprited-ai/dancing-stick-figures/blob/main/notebooks/dancing_stick_figures_colab_v0_2.ipynb)

*A hands-on notebook for readers with basic Python and neural-network familiarity. The reference `64²` route is designed for a Colab T4; `32²` is a faster sanity check.*

> **Measured reference route.** On a Tesla T4, the 64² image and video stages completed at 0.76 and 1.12 seconds per update, peaked at 6.9 and 11.3 GB, and produced a complete 120-frame rollout. Fixed sampling adds several minutes after the updates finish.

**What you'll do**
1. 👀 Look at the data — thousands of tiny dancing stick figures, each one with a hidden "skeleton" label.
2. 🎨 Train an **image** model that learns to draw a stick figure from pure noise.
3. 🎬 Reuse its spatial weights to train a **video** model, then roll it out for about 5 seconds.
4. 🤖 Diagnose the drawings and rollout: are the limbs intact, and how does the motion differ from real clips?

> **How can a computer draw from noise?** Imagine a picture covered in TV static. If you had seen a million stick figures,
> you could guess which specks of static are "probably part of a leg" and wipe the rest away — a little at a time.
> That's a *diffusion model*: it learns to un-fog a picture, step by step. We'll train one from scratch.

Before you start: **Runtime → Change runtime type → GPU** (T4 is fine).

In [ ]:
#@title 0. Setup (≈2 min) — grab the code and the small version of the dataset
import os, sys, subprocess, glob, time
V02_STARTED = time.time()
if not os.path.exists("dancing-stick-figures"):
    !git clone -q https://github.com/sprited-ai/dancing-stick-figures
%cd dancing-stick-figures
!pip install -q -r train/requirements.txt 2>&1 | tail -1
# the "mini" version: 514,800 frames at 64×64 pixels, 0.85 GB
DATA_DOWNLOAD_ATTEMPTS = 3
download_cmds = [
    ["hf", "download", "sprited/dancing-stick-figures", "--repo-type", "dataset", "--include", pattern, "--local-dir", "data/hf"]
    for pattern in ("mini/*", "motion/val-*")
]
for attempt in range(1, DATA_DOWNLOAD_ATTEMPTS + 1):
    results = [subprocess.run(cmd, text=True, capture_output=True) for cmd in download_cmds]
    mini_files = glob.glob("data/hf/mini/*.parquet"); motion_files = glob.glob("data/hf/motion/*.parquet")
    if all(result.returncode == 0 for result in results) and mini_files and motion_files: break
    print(f"dataset download attempt {attempt} did not finish; retrying")
else:
    raise RuntimeError("Dataset download did not produce mini parquet files after three attempts")
print(len(mini_files), "mini shards,", len(motion_files), "motion shard(s)")
import torch; print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE — switch the runtime to GPU!")

#@title 0b. Choose a resolution
IMAGE_SIZE = "64" #@param ["32", "64"]
IMAGE_SIZE = int(IMAGE_SIZE)
BASE_CHANNELS, NEW_FRAMES, CONTEXT_FRAMES = 64, 8, 8  # the reference backbone stays fixed
IMAGE_BATCH = 128 if IMAGE_SIZE == 32 else 64
VIDEO_BATCH = 8 if IMAGE_SIZE == 32 else 4
RUN_TAG = f"{IMAGE_SIZE}px"
IMAGE_RUN, VIDEO_RUN = f"runs/img_{RUN_TAG}", f"runs/vid_{RUN_TAG}"
print(f"resolution={IMAGE_SIZE}² · reference backbone: ch64, 8 context + 8 new frames")
print("32² is a faster sanity check; 64² is the reference setting.")

## 🧩 The fixed reference backbone

The same **factorised 3D UNet** becomes an image model when `T=1` and a video model when `T>1`. Its spatial path learns what a figure looks like; temporal attention compares the same features across frames and learns what moves.

```text
noisy RGBA frames [4, T, size, size]
             │  + diffusion-time embedding + pooled T5 prompt embedding
             ▼
down path:  size → size/2 → size/4 → size/8
             │  spatial ResBlocks + spatial/temporal attention
             ▼
          bottleneck
             │  skip connections carry fine limb details
             ▼
up path:    size/8 → size/4 → size/2 → size
             │
             ▼
predicted denoising velocity [4, T, size, size]
```

The **same code** is trained at `T=1` for images and at `T>1` for video, making the change from spatial learning to temporal learning visible without changing model families between stages. Factorising attention into within-frame and across-time operations also makes those two jobs explicit. Frozen T5-small token embeddings are mean-pooled and added through the time-conditioning path.

The sparse, low-resolution frames make direct pixel training with a compact UNet practical on a conventional GPU.

The result is an inspectable reference model whose image and video stages can be trained and diagnosed end to end.

The reference exercise keeps the backbone fixed and only lets you choose `32²` for a quick sanity check or `64²` for the reference run. The next cell prints the exact PyTorch classes used by both training stages. You do not need to edit them; the source is available for readers who later want to study it.


In [ ]:
#@title Read the actual backbone and image-to-video initialization code (optional)
import inspect
from train.video_ddpm import ResBlock, Attn, UNet3D, initialize_video_input
for component in (ResBlock, Attn, UNet3D, initialize_video_input):
    print(f"\n# --- {component.__name__} ---")
    print(inspect.getsource(component))
print("To experiment later, edit train/video_ddpm.py in Colab's Files pane. The lesson below uses the reference code unchanged.")

## 1 · 👀 Look at the data

Every dancer here is a **3D skeleton** (27 joints, like a video-game character) that a computer moved to a text prompt such as
*"A person does the running man dance"*, then photographed with a virtual camera. We keep the photo **and** the skeleton,
so we always know where every arm and leg *really* is. Colours tell body parts apart: red/orange = left arm, blue/cyan = right arm,
purple/pink = left leg, green = right leg. Head and body are black.

In [ ]:
#@title Contact sheet — 32 random frames with their prompts
import pyarrow.parquet as pq, glob, io, random, textwrap, numpy as np
from PIL import Image, ImageDraw
files = sorted(glob.glob("data/hf/mini/train-*.parquet")); random.seed(1)
row_counts = [pq.read_metadata(f).num_rows for f in files]
cumulative_rows = np.cumsum(row_counts); frame_candidates = range(int(cumulative_rows[-1]))
selected_frames = random.sample(frame_candidates, 32)
W, pad, lab, cols = 96, 6, 44, 8
sheet = Image.new("RGB", (cols*(W+pad)+pad, 4*(W+lab+pad)+pad), "white"); d = ImageDraw.Draw(sheet)
def over_white(b):
    im = Image.open(io.BytesIO(b)).convert("RGBA"); bg = Image.new("RGBA", im.size, "white"); bg.alpha_composite(im); return bg.convert("RGB")
tables = {}
for k, global_i in enumerate(selected_frames):
    shard_i = int(np.searchsorted(cumulative_rows, global_i, side="right"))
    local_i = global_i - (int(cumulative_rows[shard_i - 1]) if shard_i else 0)
    if shard_i not in tables: tables[shard_i] = pq.read_table(files[shard_i], columns=["color", "text", "group"])
    t = tables[shard_i]; r, c = divmod(k, cols); x = pad + c*(W+pad); y = pad + r*(W+lab+pad)
    sheet.paste(over_white(t.column("color")[local_i].as_py()["bytes"]).resize((W, W), Image.NEAREST), (x, y))
    d.text((x, y+W+2), t.column("group")[local_i].as_py(), fill="black")
    for li, line in enumerate(textwrap.wrap(t.column("text")[local_i].as_py(), 18)[:3]): d.text((x, y+W+13+10*li), line, fill=(90,90,90))
sheet

In [ ]:
#@title The hidden skeleton — draw the 27 joints on top of a frame
from IPython.display import display
t = pq.read_table(files[0], columns=["color", "joint_xy", "joint_visible", "text"])
i = 40
im = over_white(t.column("color")[i].as_py()["bytes"]).resize((256, 256), Image.NEAREST)
xy = np.frombuffer(t.column("joint_xy")[i].as_py(), np.float32).reshape(27, 2) * 256      # positions are stored as fractions of the image
vis = np.frombuffer(t.column("joint_visible")[i].as_py(), np.uint8)
sys.path.insert(0, "."); from generator.skeleton import NAMES, PARENT
dd = ImageDraw.Draw(im)
for j, n in enumerate(NAMES):
    p = PARENT.get(n)
    if p: dd.line([tuple(xy[j]), tuple(xy[NAMES.index(p)])], fill=(255,255,255), width=3); dd.line([tuple(xy[j]), tuple(xy[NAMES.index(p)])], fill=(0,0,0), width=1)
for j in range(27): x_, y_ = xy[j]; dd.ellipse([x_-3, y_-3, x_+3, y_+3], fill=(0,200,0) if vis[j] else (220,0,0))
print(t.column("text")[i].as_py(), "— green dots: joints the camera can see, red: hidden behind the body")
display(im)

In [ ]:
#@title Where did the dancer walk? — the root (hip) path of one clip, from the `motion` config
import matplotlib.pyplot as plt
if not glob.glob("data/hf/motion/val-*.parquet"):
    !hf download sprited/dancing-stick-figures --repo-type dataset --include "motion/val-*" --local-dir data/hf 2>&1 | tail -2
mo = pq.read_table(glob.glob("data/hf/motion/val-*.parquet")[0])
for k in range(4):
    r = mo.slice(k, 1).to_pylist()[0]; T = r["n_frames"]
    P = np.frombuffer(r["posed_joints"], np.float32).reshape(T, 27, 3)     # world coordinates in metres, 27 joints
    plt.plot(P[:, 0, 0], P[:, 0, 2], label=r["text"][:38])                # joint 0 = Hips, seen from above (x, z)
plt.axis("equal"); plt.xlabel("x (m)"); plt.ylabel("z (m)"); plt.title("Hip path seen from above, 6 seconds"); plt.legend(fontsize=7); plt.show()

## 2 · 🎨 Train an image model

First we unpack all frames into one big file so the GPU can read them fast (≈3 min). Then we train a small **UNet**
(a neural network shaped like a "U": it zooms out to see the whole figure, then zooms back in to draw fine lines).

Every training step the computer:
1. takes a real frame, 2. adds a random amount of static, 3. tries to guess what to remove, 4. gets told how wrong it was.
Do that 2,000 times and it starts drawing figures. Watch the samples appear every 500 steps. On a Tesla T4, optimization runs at 0.76 seconds per update (about 25 minutes for 2,000 updates) and peaks at 6.9 GB; validation and the four fixed sample sheets add extra time.

In [ ]:
#@title Unpack frames (train + validation) into a fast cache
!python -m train.cache --data data/hf/mini --out data/cache --splits train,val --threads 4 2>&1 | grep -v shards

In [ ]:
#@title Train the image model — 2,000 steps
import time
STEPS = 2000  #@param {type:"integer"}
image_started = time.perf_counter()
!python -m train.video_ddpm --cache data/cache --out $IMAGE_RUN --size $IMAGE_SIZE --ch $BASE_CHANNELS --frames 1 --batch $IMAGE_BATCH --steps $STEPS --sample_every 500 --val_every 500 --amp fp16 --workers 2 --cond text 2>&1 | grep --line-buffered -v Warning | grep --line-buffered "^cached\|^step\|wrote\|params\|Error\|Traceback"
IMAGE_WALL_SECONDS = time.perf_counter() - image_started
print(f"image stage wall time: {IMAGE_WALL_SECONDS / 60:.1f} min")


In [ ]:
#@title Look at what it learned — 64 drawings from pure noise, at each checkpoint
for f in sorted(glob.glob(f"{IMAGE_RUN}/sample_0*.png")):
    if "raw" in f: continue
    print(f.split("/")[-1]); display(Image.open(f).resize((512, 512), Image.NEAREST))

> **Why do early samples look like blobs?** At step 500 the model has only seen ~32,000 frames. It knows "dark blob on
> top, colours below" but not the exact rules yet. By 2,000 steps limbs appear (colours still a bit messy — that is normal). Our longer reference checkpoint saw many more frames
> and produces substantially cleaner limb structure, although it still has measurable errors and is not identical to real data. Later cells download longer checkpoints only for comparison; your video model starts from the 2,000-step image model you train here.

## 3 · 🎬 Make it move — train a video generator that dances for 5 seconds

A video is just pictures in a row. We **start from the image model you trained above**, then teach it what changes
between frames. The spatial weights transfer directly because both stages use your chosen resolution and backbone
width. New temporal and context-input weights start at zero, so the transferred image behaviour is preserved before
the video stage learns motion.

> **Prompt conditioning.** Both stages use the complete motion prompt through a frozen T5-small text encoder. The encoder supplies a conditioning signal; the factorized UNet still learns the rendered figure and its motion from this dataset. You will type a prompt after training and generate a new rollout. The current model is *prompt-conditioned*; we will call it verified prompt-following only after a separate adherence evaluation.

> **Why no Video VAE here?** Large video systems usually compress videos into a latent space first. At `32²`–`64²`,
> we can train directly on pixels, so every generated error remains visible and comparable with the exact dataset labels.
> This keeps the first lesson to one image model and one video model, without adding VAE reconstruction artifacts.
> A separate optional lesson is planned for Video VAE reconstruction and latent-space video generation.

One more trick makes the dances long: the model looks at a sliding window containing `CONTEXT_FRAMES` frames it
already drew and `NEW_FRAMES` frames it must draw. It feeds its newest frames back as context and continues — like
writing a story one sentence at a time while re-reading the previous sentence. This is called *autoregressive
diffusion*. v0.2 keeps the 8-frame context and 8-frame continuation window fixed so every run uses the same reference model.

At the final step, the training log pauses while the notebook generates fixed samples and a rollout. Sampling is intentionally deferred until training finishes because a multi-second diffusion preview takes several minutes on a T4. The measured video optimization rate is 1.12 seconds per update (about 22 minutes for 1,200 updates), with an 11.3 GB peak; validation and the 120-frame rollout follow those updates.

In [ ]:
#@title Train a 5-second video model on top of your image model
import math
INIT_CKPT = f"{IMAGE_RUN}/ckpt.pt"  # always continue from the image model trained above
assert os.path.exists(INIT_CKPT), f"Missing {INIT_CKPT}; finish image training first."
VSTEPS = 1200  #@param {type:"integer"}
ROLLOUT_CHUNKS = max(2, math.ceil((50 - (CONTEXT_FRAMES + NEW_FRAMES)) / NEW_FRAMES) + 1)
video_started = time.perf_counter()
!python -m train.video_ddpm --cache data/cache --out $VIDEO_RUN --size $IMAGE_SIZE --ch $BASE_CHANNELS --frames $NEW_FRAMES --ar_ctx $CONTEXT_FRAMES --stride 2 --batch $VIDEO_BATCH --accum 2 --steps $VSTEPS --sample_every $VSTEPS --val_every 400 --amp fp16 --workers 2 --rollout $ROLLOUT_CHUNKS --init $INIT_CKPT --cond text 2>&1 | grep --line-buffered -v Warning | grep --line-buffered "^cached\|^init\|^step\|wrote\|params\|Error\|Traceback"
VIDEO_WALL_SECONDS = time.perf_counter() - video_started
print(f"video stage wall time: {VIDEO_WALL_SECONDS / 60:.1f} min")


In [ ]:
#@title Type a prompt, then watch your model dance for 5 seconds
from IPython.display import Image as IPImage
PROMPT = "a stick figure runs forward with swinging arms"  #@param {type:"string"}
MINE_GIF = f"out/mine_prompt_{RUN_TAG}.gif"
!python scripts/rollout.py --ckpt $VIDEO_RUN/ckpt.pt --seconds 5 --n 4 --prompt "$PROMPT" --cfg 3 --out $MINE_GIF --score --cache data/cache 2>&1 | grep -v FutureWarning
print(f"yours — prompt: {PROMPT!r}"); display(IPImage(filename=MINE_GIF))

> The figures were "born" knowing how to stand (from the image model). Everything that changes between frames —
> arm swings, weight shifts — was learned in the video stage, and the 5-second length comes from chaining your chosen
> `NEW_FRAMES`-sized chunks. If yours mostly wiggle in place, train longer or compare the 32² sanity setting with the
> 64² reference setting. Look closely for tiny boundary "seams" where one chunk hands over to the next.

## 4 · 🤖 Diagnose visible failures

How can we describe a visible failure without asking a human to inspect every frame? Because every body part has its own
colour, a small structural evaluator can **count**: is there exactly one red arm? Is the pink shin touching the purple thigh? Are the
colours clean or smeared? These measurements diagnose specific properties; they are not a single overall grade.

- **tvr** — *topology violation rate*: a coloured limb is missing or split into more than one piece
- **lie** — *limb-identity error*: colours that should meet (for example, upper arm and forearm) fail to touch
- **clean** — share of drawings with zero mistakes

One catch: real dancers sometimes hide an arm behind their body — the robot counts that as "missing" too! So we always
compare against **real-reference** frames. A lower score is not automatically better than the real reference: an unnaturally simple or frozen figure can also be easy for this colour-based checker.

The rollout cell also reports centroid speed, acceleration, motion fraction, and angular jerk beside the same measurements on real clips. These numbers describe different visible motion behaviours; none is a single overall quality grade.

In [ ]:
#@title Measure visible structure in your image model, real frames, and the reference checkpoint
import json
MINE_JSON = f"out/mine_{RUN_TAG}.json"
!python -m eval.score_images --ckpt $IMAGE_RUN/ckpt.pt --cache data/cache --n 128 --steps 30 --out $MINE_JSON 2>&1 | tail -1
mine = json.load(open(MINE_JSON)); ref = None
if IMAGE_SIZE == 64 and BASE_CHANNELS == 64:
    if not os.path.exists("ckpts/unet_img64.pt"):
        !hf download sprited/dancing-stick-figures-baselines unet_img64.pt --local-dir ckpts 2>&1 | tail -2
    !python -m eval.score_images --ckpt ckpts/unet_img64.pt --cache data/cache --n 128 --steps 30 --out out/ref.json 2>&1 | tail -1
    ref = json.load(open("out/ref.json"))
print(f"{'':22s} {'lie':>6s} {'tvr':>6s} {'clean':>6s}")
print(f"{'your model':22s} {mine['lie']:6.3f} {mine['tvr']:6.3f} {mine['clean_frac']:6.2f}")
if ref: print(f"{'reference (100k steps)':22s} {ref['lie']:6.3f} {ref['tvr']:6.3f} {ref['clean_frac']:6.2f}")
print(f"{'real reference':22s} {mine['floor']['lie']:6.3f} {mine['floor']['tvr']:6.3f} {mine['floor']['clean_frac']:6.2f}")

In [ ]:
#@title Verification record — timings, memory, and required artifacts
from pathlib import Path
import re
def peak_gb(path):
    values = [float(x) for x in re.findall(r"peak ([0-9.]+)GB", Path(path).read_text())]
    return max(values) if values else float("nan")
required = [f"{IMAGE_RUN}/ckpt.pt", f"{VIDEO_RUN}/ckpt.pt", MINE_GIF, MINE_JSON]
missing = [path for path in required if not Path(path).exists()]
assert not missing, f"missing required artifacts: {missing}"
verification = {
    "image_wall_seconds": IMAGE_WALL_SECONDS,
    "video_wall_seconds": VIDEO_WALL_SECONDS,
    "image_peak_gb": peak_gb(IMAGE_RUN + "/log.txt"),
    "video_peak_gb": peak_gb(VIDEO_RUN + "/log.txt"),
    "total_wall_seconds": time.time() - V02_STARTED,
    "gpu": torch.cuda.get_device_name(0),
    "resolution": IMAGE_SIZE,
    "image_steps": STEPS,
    "video_steps": VSTEPS,
}
Path("out").mkdir(exist_ok=True)
Path("out/v02_completion.json").write_text(json.dumps(verification, indent=2))
print(f"V02_IMAGE_PEAK_GB={verification['image_peak_gb']:.1f}")
print(f"V02_VIDEO_PEAK_GB={verification['video_peak_gb']:.1f}")
print(f"V02_TOTAL_WALL_SECONDS={verification['total_wall_seconds']:.1f}")
print("V02_GPU=" + verification["gpu"])
print("V02_COMPLETE=1")

## 🚀 Where to go next

- Train the same reference model longer (`STEPS = 5000`) and compare **lie/tvr** with the real reference.
- Read the printed `UNet3D` source when you want to understand exactly what the model is doing. The notebook does not require architecture changes.
- Use the exact state labels for another learning task, such as estimating a skeleton from an image.

Dataset: https://huggingface.co/datasets/sprited/dancing-stick-figures · Code: https://github.com/sprited-ai/dancing-stick-figures · Made by Sprited.